# Linkography: Unfolding the Design Process - Gabriela Goldschmidt 
## This is implementing the method into the current project

Unsure at the moment to what extend we are implementing Goldschmidt's linkography.

Much of it is in design space.... we are trying to add it in the sense of computational social science

Doable for sure...

## this was a very early code chunk to get in the formula - data input not set etc. 

In [2]:
# ================================================================
# Linkography (lite): utterance→utterance links inside ONE session
# - Links if cosine_sim(text_i, text_j) >= TAU and i<j (forward link)
#   + optional boost if they share ≥1 annotation code
# - Saves to your chosen folder:
#     figures/: arc diagram + upper-triangular link matrix
#     linkography-output/: link list CSV + per-utterance centrality CSV
# - Prints a compact metrics summary
# ================================================================
from __future__ import annotations
from typing import List, Tuple, Optional
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path
import re

# -------------------- CONFIG --------------------
# Similarity / linking
TAU_SIM            = 0.30     # cosine similarity threshold to create a link
ANN_OVERLAP_BONUS  = 0.05     # add to similarity if any annotation overlapped
MIN_GAP            = 1        # require at least this many utterances between i and j
MAX_SPAN           = None     # if set (e.g., 25), only consider links where j - i <= MAX_SPAN
USE_ANN_OVERLAP    = True     # use annotation overlap bump if annotations present
MAX_MOVES          = None     # if set (e.g., 400), truncate very long sessions for speed

# Your writable project folder (✅ you said to use this)
BASE_OUT = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/linkography/l-testing/outputs")

OUTDIR = BASE_OUT / "figures"
TABDIR = BASE_OUT / "linkography-output"
OUTDIR.mkdir(parents=True, exist_ok=True)
TABDIR.mkdir(parents=True, exist_ok=True)

# -------------------- HELPERS --------------------
def _split_ann(x) -> set:
    """Support list/tuple or comma/semicolon-separated strings; else empty set."""
    if x is None or (isinstance(x, float) and np.isnan(x)): 
        return set()
    if isinstance(x, (list, tuple)): 
        return {str(a).strip() for a in x if str(a).strip()}
    return {s.strip() for s in re.split(r"[,;]", str(x)) if s.strip()}

def _auto_session_key(df: pd.DataFrame) -> str:
    """Pick a session column that identifies ONE meeting (best-effort)."""
    for cand in ["session_id", "meeting_id", "conference_year", "global_session"]:
        if cand in df.columns:
            return cand
    raise AssertionError("No session identifier found. Add one of: session_id, meeting_id, conference_year, global_session.")

def _prep_session(df_session: pd.DataFrame) -> pd.DataFrame:
    """Sort, index, normalize minimal columns for one session slice."""
    need = {"utterance_id", "timestamp", "speaker", "transcript"}
    missing = [c for c in need if c not in df_session.columns]
    assert not missing, f"Missing required columns: {missing}"
    d = df_session.sort_values(["timestamp","utterance_id"]).copy()
    if MAX_MOVES is not None and len(d) > MAX_MOVES:
        d = d.iloc[:MAX_MOVES].copy()
    d["u_idx"] = np.arange(len(d))  # 0..N-1 for plotting
    d["transcript"] = d["transcript"].fillna("").astype(str)

    if "annotations" in d.columns:
        d["ann_set"] = d["annotations"].apply(_split_ann)
    else:
        d["ann_set"] = [set()]*len(d)

    return d

def _tfidf_cosine(texts: List[str]) -> np.ndarray:
    """TF-IDF with unigrams+bigrams, sensible df bounds; returns dense cosine matrix."""
    if len(texts) <= 1:
        return np.zeros((len(texts), len(texts)), dtype=float)
    vect = TfidfVectorizer(min_df=1, max_df=0.95, ngram_range=(1,2))
    X = vect.fit_transform(texts)
    return cosine_similarity(X, X, dense_output=True)

def _build_links(d: pd.DataFrame) -> Tuple[pd.DataFrame, np.ndarray]:
    """Compute pairwise sims; create forward links passing threshold (with optional ann bonus)."""
    sims = _tfidf_cosine(d["transcript"].tolist())
    N = len(d)
    links = []
    for i in range(N):
        for j in range(i+1, N):
            if j - i < MIN_GAP:
                continue
            if MAX_SPAN is not None and (j - i) > MAX_SPAN:
                continue
            s = sims[i, j]
            if USE_ANN_OVERLAP and (d.at[i,"ann_set"] or d.at[j,"ann_set"]):
                if d.at[i,"ann_set"] & d.at[j,"ann_set"]:
                    s = s + ANN_OVERLAP_BONUS
            if s >= TAU_SIM:
                links.append({
                    "src_idx": i, "dst_idx": j,
                    "src_uid": d.at[i,"utterance_id"], "dst_uid": d.at[j,"utterance_id"],
                    "src_speaker": d.at[i,"speaker"],    "dst_speaker": d.at[j,"speaker"],
                    "sim": float(s),
                    "span": int(j - i),
                    "ann_overlap": int(bool(d.at[i,"ann_set"] & d.at[j,"ann_set"]))
                })
    L = pd.DataFrame(links)
    return L, sims

def _metrics(d: pd.DataFrame, L: pd.DataFrame) -> dict:
    """Basic linkography descriptors + identify top betweenness 'critical moves'."""
    N = len(d)
    possible = N*(N-1)/2  # forward links only
    Lcnt = len(L)
    density = Lcnt/possible if possible > 0 else 0.0
    long_span_frac = L["span"].ge(max(2, int(0.25*N))).mean() if Lcnt>0 else 0.0
    same_speaker_frac = (L["src_speaker"]==L["dst_speaker"]).mean() if Lcnt>0 else 0.0

    # DiGraph where edges go forward in time
    G = nx.DiGraph()
    G.add_nodes_from(range(N))
    for _, r in L.iterrows():
        G.add_edge(int(r["src_idx"]), int(r["dst_idx"]), weight=float(r["sim"]))

    indeg  = np.array([G.in_degree(i) for i in range(N)], dtype=float) if N>0 else np.array([0.0])
    outdeg = np.array([G.out_degree(i) for i in range(N)], dtype=float) if N>0 else np.array([0.0])

    btw = nx.betweenness_centrality(G, normalized=True) if N>2 else {i: 0.0 for i in range(N)}
    crit_top3 = sorted(btw.items(), key=lambda kv: kv[1], reverse=True)[:3]

    return dict(
        utterances=N,
        links=Lcnt,
        density=round(density, 4),
        mean_in_degree=round(indeg.mean() if N>0 else 0, 3),
        mean_out_degree=round(outdeg.mean() if N>0 else 0, 3),
        long_span_frac=round(float(long_span_frac), 3),
        same_speaker_link_frac=round(float(same_speaker_frac), 3),
        critical_moves=[(int(k), float(v)) for k,v in crit_top3]
    )

def _plot_arc(d: pd.DataFrame, L: pd.DataFrame, title: str, outpath: Path):
    """Classic linkograph arc plot."""
    N = len(d)
    if N == 0:
        return
    y = np.zeros(N)
    x = d["u_idx"].values
    fig, ax = plt.subplots(figsize=(10, 2.5))
    ax.scatter(x, y, s=8, color="black", zorder=3)
    for _, r in L.iterrows():
        i, j = int(r["src_idx"]), int(r["dst_idx"])
        rad = 0.08*(j - i)
        t = np.linspace(0, np.pi, 48)
        xx = x[i] + (x[j]-x[i])*(t/np.pi)
        yy = rad*np.sin(t)
        ax.plot(xx, yy, alpha=0.28, linewidth=0.8, zorder=2)
    ax.set_yticks([])
    step = max(1, N//12)
    ax.set_xticks(x[::step])
    ax.set_xticklabels(d["utterance_id"].astype(str).values[::step], rotation=90)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)

def _plot_upper_tri(L: pd.DataFrame, N: int, title: str, outpath: Path):
    """Upper-triangular link matrix heatmap (i→j sim)."""
    if N == 0:
        return
    M = np.zeros((N, N))
    for _, r in L.iterrows():
        M[int(r["src_idx"]), int(r["dst_idx"])] = r["sim"]
    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(M, cmap="viridis", origin="lower", interpolation="nearest")
    ax.set_title(title)
    ax.set_xlabel("j (future move)"); ax.set_ylabel("i (earlier move)")
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel("similarity", rotation=270, labelpad=12)
    fig.tight_layout()
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)

# -------------------- MAIN ENTRY --------------------
def run_linkography_for_session(df_all: pd.DataFrame,
                                session_id_value,
                                session_col: Optional[str]=None,
                                save_tag: Optional[str]=None) -> dict:
    """
    df_all            : full long dataframe containing at least one complete session
    session_id_value  : the value identifying the ONE session to analyze
    session_col       : column name of session id (auto-detected if None)
    save_tag          : string to use in output filenames (defaults to str(session_id_value))
    """
    # Pick session column
    sess_col = session_col or _auto_session_key(df_all)
    dsub = df_all[df_all[sess_col] == session_id_value].copy()
    assert len(dsub) > 0, f"No rows found for {sess_col}=={session_id_value!r}."

    # Prep and build links
    d = _prep_session(dsub)
    L, sims = _build_links(d)

    # Filenames
    tag = save_tag or str(session_id_value)
    links_csv = TABDIR / f"{tag}_links.csv"
    peru_csv  = TABDIR / f"{tag}_per_utterance_degree.csv"
    arc_png   = OUTDIR / f"{tag}_arc.png"
    mat_png   = OUTDIR / f"{tag}_matrix.png"

    # Save links + per-utterance degree
    L.to_csv(links_csv, index=False)
    # per-utterance "link degree" (in+out) across directed forward links
    if len(L) > 0:
        peru = (
            L.melt(value_vars=["src_idx","dst_idx"], value_name="idx")
             .drop(columns=["variable"])
             .groupby("idx").size().reindex(range(len(d)), fill_value=0)
             .rename("link_degree").reset_index()
        )
    else:
        peru = pd.DataFrame({"idx": np.arange(len(d)), "link_degree": 0})
    peru["utterance_id"] = d["utterance_id"].values
    peru["speaker"]      = d["speaker"].values
    peru.to_csv(peru_csv, index=False)

    # Metrics
    met = _metrics(d, L)

    # Plots
    _plot_arc(d, L, f"Linkograph — {tag}", arc_png)
    _plot_upper_tri(L, len(d), f"Link matrix — {tag}", mat_png)

    # Console summary
    print("=== Linkography summary ===")
    for k,v in met.items():
        if k != "critical_moves":
            print(f"{k:24s}: {v}")
    if met["critical_moves"]:
        print("critical_moves (idx, betweenness):", met["critical_moves"])
        print("  (idx refers to 0-based order; use u_idx→utterance_id to crosswalk)")
    print(f"\nSaved:\n  {links_csv}\n  {peru_csv}\n  {arc_png}\n  {mat_png}")

    return dict(links=L, per_utterance=peru, metrics=met)

# -------------------- EXAMPLE USAGE --------------------
# Suppose your long DF is `df` (already in memory).
# If your session key is 'conference_year' (common in your data), do:
#
# sess_col = "conference_year"  # or run _auto_session_key(df) to discover
# sess_val = df[sess_col].iloc[0]   # pick a concrete session value to test
# out = run_linkography_for_session(df, session_id_value=sess_val, session_col=sess_col)
#
# If your data has 'session_id', use that instead:
# out = run_linkography_for_session(df, session_id_value=df["session_id"].iloc[0], session_col="session_id")